In [1]:
import torch 
import torch.nn as nn 
import torch.nn.functional as f

from torchvision import datasets, transforms
from torch.utils.data import DataLoader

In [ ]:
"""ACTUAL TRAINING CODE"""
device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')

# Train Function 
def train_batch(x, y, model, opt, loss_fn):
    model.train() 

    opt.zero_grad()
    batch_loss = loss_fn(model(x), y)
    batch_loss.backward()
    opt.step()

    return batch_loss.detach().cpu().numpy()

# Evaluation function
@torch.no_grad()
def accuracy(x, y, model):
    model.eval() 

    prediction = model(x) 
    argmaxes = prediction.argmax(dim=1)
    s = torch.sum((argmaxes == y).float())/len(y)
    return s.detach().cpu().numpy()


mlp = MLP().to(device)
loss_func = nn.CrossEntropyLoss()
optimizer = Adam(mlp.parameters(), lr=1e-3)
n_epochs = 5

total_params = sum(p.numel() for p in mlp.parameters())
print(f"Total parameters: {total_params}")

# Training Loop 
import numpy as np 
losses, accuracies, n_epochs = [], [], 5 

for epoch in range(n_epochs):
    print(f"Running epoch {epoch+1} of {n_epochs}")

    # Train
    epoch_losses, epoch_accuracies = [], []
    for batch in train_dl_mlp: 
        x, y = batch 
        batch_loss = train_batch(x, y, mlp, optimizer, loss_func)
        epoch_losses.append(batch_loss)
    epoch_loss = np.mean(epoch_losses)

    # Evaluate
    for batch in test_dl_mlp: 
        x, y = batch 
        batch_acc = accuracy(x, y, mlp)
        epoch_accuracies.append(batch_acc)
    epoch_acc = np.mean(epoch_accuracies)

    losses.append(epoch_loss)
    accuracies.append(epoch_acc)



# Plotting 
import matplotlib.pyplot as plt
plt.figure(figsize=(13, 3))
plt.subplot(121) 
plt.title("Training Loss value over epochs")
plt.plot(np.arange(n_epochs) + 1, losses)
plt.subplot(122)
plt.title("Training Accuracy value over epochs")
plt.plot(np.arange(n_epochs) + 1, accuracies)
plt.show()


In [2]:
from torchvision import models
model = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1)


for param in model.features.parameters():
    param.requires_grad = False

model.avgpool = nn.AdaptiveAvgPool2d(output_size=(1, 1))
model.classifier = nn.Sequential(nn.Flatten(),
nn.Linear(512, 128), nn.ReLU(), nn.Dropout(0.2),
nn.Linear(128, 1), nn.Sigmoid())

Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /Users/mingikang/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:32<00:00, 16.8MB/s] 


In [ ]:
data_path = "./Datasets"
train_data = datasets.CIFAR10(root=data_path, train=True, download=True)
test_data = datasets.CIFAR10(root=data_path, train=False, download=True)

train_loader = DataLoader(
    dataset = train_data, 
    batch_size = 16, 
    shuffle = True)

test_loader = DataLoader(
    dataset = train_data, 
    batch_size = 16, 
    shuffle = False)

        